# 14 fonctions Pandas pour le Data Cleaning

Notebook de l'article [14 fonctions Pandas que les Data Scientists utilisent 60 % du temps pour le Data Cleaning](https://moncoachdata.com/blog/14-fonctions-pandas-pour-data-science/) (MonCoachData).

On nettoie et on transforme les matchs des Coupes du Monde de football (1930-2018) scrapés sur Wikipédia. Exécute les cellules **dans l'ordre** : les trois fichiers CSV sont lus directement depuis le dépôt GitHub `moncoachdata/blog-data-science` (il faut une connexion internet).

Si tu travailles sur ton ordinateur, installe d'abord Pandas (déjà présent dans Google Colab) : `pip install pandas`.

## 1. Charger les trois jeux de données avec `read_csv`

Attendu : `(457, 4) (64, 4) (508, 4)`.

In [1]:
import pandas as pd

BASE = "https://raw.githubusercontent.com/moncoachdata/blog-data-science/main/14-fonctions-pandas-pour-data-science/"

df_historical_data = pd.read_csv(BASE + "fifa_worldcup_matches.csv")
df_fixture = pd.read_csv(BASE + "fifa_worldcup_fixture.csv")
df_missing_data = pd.read_csv(BASE + "fifa_worldcup_missing_data.csv")

In [2]:
print(df_historical_data.shape, df_fixture.shape, df_missing_data.shape)

(457, 4) (64, 4) (508, 4)


## 2. Supprimer les espaces (insécables) avec `str.strip`

Les noms d'équipes se terminent par `\xa0`, une espace insécable invisible.

In [3]:
df_fixture.head(8)

,home,score,away,year
0,Qatar,Match 1,Ecuador,2022
1,Senegal,Match 2,Netherlands,2022
2,Qatar,Match 18,Senegal,2022
3,Netherlands,Match 19,Ecuador,2022
4,Ecuador,Match 35,Senegal,2022
5,Netherlands,Match 36,Qatar,2022
6,England,Match 3,Iran,2022
7,United States,Match 4,Wales,2022


In [4]:
df_fixture["home"].head(3).tolist()

['Qatar\xa0', 'Senegal\xa0', 'Qatar\xa0']

In [5]:
df_fixture["home"] = df_fixture["home"].str.strip()
df_fixture["away"] = df_fixture["away"].str.strip()

In [6]:
df_fixture["home"].head(3).tolist()

['Qatar', 'Senegal', 'Qatar']

## 3. Valeurs manquantes : `isnull` et `dropna`

Attendu : 64 lignes vides, puis `(444, 4)` après `dropna`.

In [7]:
df_missing_data.isnull().sum()

,0
home,64
score,64
away,64
year,0


In [8]:
df_missing_data[df_missing_data["home"].isnull()].head(5)

,home,score,away,year
396,NaN,NaN,NaN,2010
397,NaN,NaN,NaN,2010
398,NaN,NaN,NaN,2010
399,NaN,NaN,NaN,2010
400,NaN,NaN,NaN,2010


In [9]:
df_missing_data = df_missing_data.dropna()
df_missing_data.shape

(444, 4)

## 4. Fusionner avec `concat` et supprimer les doublons avec `drop_duplicates`

Attendu : `(901, 4)` et 0 doublon.

In [10]:
df_historical_data = pd.concat([df_historical_data, df_missing_data], ignore_index=True)
df_historical_data.shape

(901, 4)

In [11]:
print(df_historical_data.duplicated().sum())

0


In [12]:
df_historical_data = (
    df_historical_data
    .drop_duplicates()
    .sort_values("year", kind="stable")
    .reset_index(drop=True)
)
df_historical_data.shape

(901, 4)

In [13]:
df_historical_data.tail(6)

,home,score,away,year
895,Sweden,0–2,England,2018
896,Russia,2–2 (a.e.t.),Croatia,2018
897,France,1–0,Belgium,2018
898,Croatia,2–1 (a.e.t.),England,2018
899,Belgium,2–0,England,2018
900,France,4–2,Croatia,2018


## 5. Repérer et supprimer une anomalie : `str.contains` et `drop`

Le match Suède-Autriche de 1938 est un forfait (`w/o[a]`). Attendu après suppression : `(900, 4)`.

In [14]:
df_historical_data[df_historical_data["score"].str.contains("w/o")]

,home,score,away,year
37,Sweden,w/o[a],Austria,1938


In [15]:
walkover = df_historical_data[
    df_historical_data["home"].str.contains("Sweden")
    & df_historical_data["away"].str.contains("Austria")
].index
df_historical_data = df_historical_data.drop(index=walkover)
df_historical_data.shape

(900, 4)

## 6. Corriger les incohérences avec une regex et `str.replace`

Attendu : 68 scores avec `(a.e.t.)`, puis 0 incohérence.

In [16]:
incoherences = df_historical_data[df_historical_data["score"].str.contains(r"[^\d–]")]
len(incoherences)

68

In [17]:
incoherences.head(6)

,home,score,away,year
24,Austria,3–2 (a.e.t.),France,1934
27,Italy,1–1 (a.e.t.),Spain,1934
34,Italy,2–1 (a.e.t.),Czechoslovakia,1934
35,Switzerland,1–1 (a.e.t.),Germany,1938
38,Cuba,3–3 (a.e.t.),Romania,1938
40,Italy,2–1 (a.e.t.),Norway,1938


In [18]:
df_historical_data["score"] = df_historical_data["score"].str.replace(r"[^\d–]", "", regex=True)
print(df_historical_data["score"].str.contains(r"[^\d–]").sum())

0


In [19]:
df_historical_data["home"] = df_historical_data["home"].str.strip()
df_historical_data["away"] = df_historical_data["away"].str.strip()
df_historical_data["home"].head(3).tolist()

['France', 'Argentina', 'Chile']

## 7. Séparer le score avec `str.split`

In [20]:
df_historical_data[["HomeGoals", "AwayGoals"]] = df_historical_data["score"].str.split("–", expand=True)
df_historical_data = df_historical_data.drop(columns="score")
df_historical_data.head(3)

,home,away,year,HomeGoals,AwayGoals
0,France,Mexico,1930,4,1
1,Argentina,France,1930,1,0
2,Chile,Mexico,1930,3,0


## 8. Renommer avec `rename` et convertir avec `astype`

Avec pandas 3, les colonnes de texte s'affichent `str` au lieu de `object`.

In [21]:
df_historical_data = df_historical_data.rename(
    columns={"home": "HomeTeam", "away": "AwayTeam", "year": "Year"}
)
df_historical_data.dtypes

,0
HomeTeam,object
AwayTeam,object
Year,int64
HomeGoals,object
AwayGoals,object


In [22]:
df_historical_data = df_historical_data.astype({"HomeGoals": int, "AwayGoals": int})
df_historical_data.dtypes

,0
HomeTeam,object
AwayTeam,object
Year,int64
HomeGoals,int64
AwayGoals,int64


## 9. Vérifier le résultat

Attendu : 900 matchs et 2548 buts.

In [23]:
buts = df_historical_data["HomeGoals"].sum() + df_historical_data["AwayGoals"].sum()
print("Matchs :", len(df_historical_data), "| Buts :", buts)
df_historical_data.groupby("Year").size().to_dict()

Matchs : 900 | Buts : 2548


{1930: 18,
 1934: 17,
 1938: 18,
 1950: 22,
 1954: 26,
 1958: 35,
 1962: 32,
 1966: 32,
 1970: 32,
 1974: 38,
 1978: 38,
 1982: 52,
 1986: 52,
 1990: 52,
 1994: 52,
 1998: 64,
 2002: 64,
 2006: 64,
 2010: 64,
 2014: 64,
 2018: 64}

## 10. Exporter avec `to_csv`

In [24]:
df_historical_data.to_csv("clean_fifa_worldcup_matches.csv", index=False)
df_fixture.to_csv("clean_fifa_worldcup_fixture.csv", index=False)

## Tout le code

In [25]:
import pandas as pd

BASE = "https://raw.githubusercontent.com/moncoachdata/blog-data-science/main/14-fonctions-pandas-pour-data-science/"

df_historical_data = pd.read_csv(BASE + "fifa_worldcup_matches.csv")
df_fixture = pd.read_csv(BASE + "fifa_worldcup_fixture.csv")
df_missing_data = pd.read_csv(BASE + "fifa_worldcup_missing_data.csv")

# 1. Nettoyer df_fixture
df_fixture["home"] = df_fixture["home"].str.strip()
df_fixture["away"] = df_fixture["away"].str.strip()

# 2. Supprimer les lignes vides de df_missing_data et fusionner
df_missing_data = df_missing_data.dropna()
df_historical_data = pd.concat([df_historical_data, df_missing_data], ignore_index=True)
df_historical_data = (
    df_historical_data
    .drop_duplicates()
    .sort_values("year", kind="stable")
    .reset_index(drop=True)
)

# 3. Supprimer le walkover Suède - Autriche (1938)
walkover = df_historical_data[
    df_historical_data["home"].str.contains("Sweden")
    & df_historical_data["away"].str.contains("Austria")
].index
df_historical_data = df_historical_data.drop(index=walkover)

# 4. Corriger les scores et les noms d'équipes
df_historical_data["score"] = df_historical_data["score"].str.replace(r"[^\d–]", "", regex=True)
df_historical_data["home"] = df_historical_data["home"].str.strip()
df_historical_data["away"] = df_historical_data["away"].str.strip()

# 5. Transformer : séparer le score, renommer, convertir
df_historical_data[["HomeGoals", "AwayGoals"]] = df_historical_data["score"].str.split("–", expand=True)
df_historical_data = df_historical_data.drop(columns="score")
df_historical_data = df_historical_data.rename(
    columns={"home": "HomeTeam", "away": "AwayTeam", "year": "Year"}
)
df_historical_data = df_historical_data.astype({"HomeGoals": int, "AwayGoals": int})

# 6. Exporter
df_historical_data.to_csv("clean_fifa_worldcup_matches.csv", index=False)
df_fixture.to_csv("clean_fifa_worldcup_fixture.csv", index=False)
df_historical_data.head()

,HomeTeam,AwayTeam,Year,HomeGoals,AwayGoals
0,France,Mexico,1930,4,1
1,Argentina,France,1930,1,0
2,Chile,Mexico,1930,3,0
3,Chile,France,1930,1,0
4,Argentina,Mexico,1930,6,3
